# 11 — Solvation Effects: PCM, COSMO & pKa Prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/11_solvation_effects.ipynb)

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- Understand implicit solvation models (PCM, COSMO, SMD)
- Run ddCOSMO solvation calculations with PySCF
- Compute solvation free energies ΔG_solv for small molecules
- Estimate pKa using a thermodynamic cycle
- Interpret how the dielectric constant controls solvation strength
- Know when to use explicit vs implicit solvation

## 1. Theory: Implicit Solvation Models

### 1.1 Born Model (Spherical Cavity)

The simplest model places charge $q$ in a spherical cavity of radius $a$ in a dielectric
of permittivity $\varepsilon$:

$$\Delta G_{\rm Born} = -\frac{q^2}{2a}\left(1 - \frac{1}{\varepsilon}\right)$$

For water ($\varepsilon \approx 78.4$), $\Delta G_{\rm Born}$ captures most ionic solvation
energy but neglects molecular shape effects.

### 1.2 Polarizable Continuum Model (PCM)

PCM builds a **molecular-shaped cavity** (van der Waals or solvent-excluding surface)
and places apparent surface charges $\sigma(\mathbf{s})$ at the boundary to screen the
solute's electric field:

$$V_{\rm react}(\mathbf{r}) = \int_{\partial\Omega}
  \frac{\sigma(\mathbf{s})}{|\mathbf{r}-\mathbf{s}|}\,d\mathbf{s}$$

The solute Hamiltonian becomes $\hat{H} = \hat{H}_0 + \hat{V}_{\rm react}$, solved
self-consistently.

### 1.3 COSMO and ddCOSMO

**COSMO** approximates the solvent as a conductor ($\varepsilon\to\infty$) then rescales:

$$q^* = -\frac{\varepsilon-1}{\varepsilon+0.5}\,q_{\rm conductor}$$

PySCF implements **ddCOSMO** (domain-decomposition COSMO), which scales as $O(N)$
with system size.

### 1.4 Thermodynamic Cycle for pKa

$$\mathrm{HA_{(aq)}} \rightleftharpoons \mathrm{H^+_{(aq)}} + \mathrm{A^-_{(aq)}}$$

$$\Delta G_{\rm aq}^{\rm deprot} = \Delta G_{\rm gas}^{\rm deprot}
  + \Delta G_{\rm solv}(A^-) + \Delta G_{\rm solv}(H^+)
  - \Delta G_{\rm solv}(HA)$$

$$\mathrm{p}K_a = \frac{\Delta G_{\rm aq}^{\rm deprot}}{RT\ln 10}$$

where $\Delta G_{\rm solv}(H^+) = -265.9$ kcal/mol (experimental absolute proton solvation).

In [ ]:
%%time
# =============================================================================
# Ch121a: Quantum Chemistry & DFT — Notebook 11: Solvation Effects
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
from pyscf import gto, dft, solvent

HARTREE2KCAL = 627.5095

def run_dft_gas(atom_str, basis='def2-SVP', xc='b3lyp', charge=0, spin=0):
    mol = gto.M(atom=atom_str, basis=basis, charge=charge, spin=spin, verbose=0)
    mf = dft.RKS(mol); mf.xc = xc
    mf.kernel()
    return mf.e_tot

def run_ddcosmo(atom_str, basis='def2-SVP', xc='b3lyp', eps=78.4, charge=0, spin=0):
    mol = gto.M(atom=atom_str, basis=basis, charge=charge, spin=spin, verbose=0)
    mf_base = dft.RKS(mol); mf_base.xc = xc
    mf_solv = solvent.ddCOSMO(mf_base)
    mf_solv.with_solvent.eps = eps
    mf_solv.kernel()
    return mf_solv.e_tot

# ------------------------------------------------------------------
# Solvation Free Energies in Water (ε = 78.4)
# ------------------------------------------------------------------
molecules = {
    'H2O':   'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    'NH3':   'N 0 0 0.116; H 0 0.939 -0.271; H 0.813 -0.470 -0.271; H -0.813 -0.470 -0.271',
    'HF':    'H 0 0 0; F 0 0 0.917',
    'CH3OH': 'C 0 0 0; O 1.43 0 0; H -0.36 0.89 0; H -0.36 -0.89 0; H -0.36 0 0.89; H 1.85 0.96 0',
}

print(f"{'Molecule':>8}  {'E_gas (Eh)':>14}  {'E_solv (Eh)':>14}  {'ΔG_solv (kcal/mol)':>20}")
print("-" * 62)
dG_results = {}
for name, atom in molecules.items():
    e_gas = run_dft_gas(atom)
    e_solv = run_ddcosmo(atom)
    dG = (e_solv - e_gas) * HARTREE2KCAL
    dG_results[name] = dG
    print(f"{name:>8}  {e_gas:14.6f}  {e_solv:14.6f}  {dG:20.2f}")

In [ ]:
# ------------------------------------------------------------------
# Bar Chart: Solvation Energies
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
names = list(dG_results.keys())
vals = [dG_results[n] for n in names]
colors = ['steelblue' if v < 0 else 'coral' for v in vals]
bars = ax.bar(names, vals, color=colors, edgecolor='black', linewidth=0.8)
ax.axhline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, val - 0.15 * abs(val),
            f'{val:.1f}', ha='center', va='top', fontsize=10)
ax.set_ylabel('ΔG_solv (kcal/mol)', fontsize=12)
ax.set_title('Solvation Free Energies in Water\n(ddCOSMO / B3LYP / def2-SVP)', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
%%time
# ------------------------------------------------------------------
# Effect of Dielectric Constant on Solvation Energy of H2O
# ------------------------------------------------------------------
water_atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469'
e_gas_w = run_dft_gas(water_atom)

eps_vals = [2.0, 4.81, 10.0, 20.0, 35.7, 46.7, 78.4]
labels   = ['hexane\n(2.0)', 'CHCl3\n(4.8)', 'DCM\n(10)', 'acetone-like\n(20)',
            'DMSO-like\n(36)', 'DMF-like\n(47)', 'water\n(78)']
dG_eps = []
for eps in eps_vals:
    e_s = run_ddcosmo(water_atom, eps=eps)
    dG_eps.append((e_s - e_gas_w) * HARTREE2KCAL)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(eps_vals, dG_eps, 'o-', color='steelblue', linewidth=2, markersize=8)
for x, y, lbl in zip(eps_vals, dG_eps, labels):
    ax.annotate(lbl, xy=(x, y), xytext=(x, y - 0.08), fontsize=8, ha='center')
ax.set_xlabel('Dielectric constant ε', fontsize=12)
ax.set_ylabel('ΔG_solv (kcal/mol)', fontsize=12)
ax.set_title('Solvation Energy of H₂O vs Dielectric Constant', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
%%time
# ------------------------------------------------------------------
# Approximate pKa of Acetic Acid via Thermodynamic Cycle
# ------------------------------------------------------------------
# ΔG_solv(H+) = -265.9 kcal/mol (Tissandier et al., experimental)
DG_SOLV_HPLUS = -265.9

HA = 'C -0.56 0 0; C 0.56 0 0; O 1.19 1.07 0; O 1.19 -1.07 0; H -0.56 1.03 0; H -0.56 -1.03 0; H -1.59 0 0'
A  = 'C -0.56 0 0; C 0.56 0 0; O 1.19 1.07 0; O 1.19 -1.07 0; H -0.56 1.03 0; H -0.56 -1.03 0; H -1.59 0 0'

# Use a simpler acetic acid geometry
HA_atom = 'C 0 0 0; O 0 1.22 0; O 1.34 -0.04 0; H 1.67 0.82 0; H -1.02 0 0; H 0.27 0 1.03; H 0.27 0 -1.03'
A_atom  = 'C 0 0 0; O 0 1.25 0; O 1.25 0 0; H -1.02 0 0; H 0.27 0 1.03; H 0.27 0 -1.03'

e_HA_gas = run_dft_gas(HA_atom, charge=0)
e_A_gas  = run_dft_gas(A_atom, charge=-1)
e_H_gas  = 0.0  # proton has no electrons; nuclear repulsion = 0 for bare H+

dG_gas = (e_A_gas + e_H_gas - e_HA_gas) * HARTREE2KCAL

e_HA_solv = run_ddcosmo(HA_atom, charge=0)
e_A_solv  = run_ddcosmo(A_atom, charge=-1)
dG_solv_HA = (e_HA_solv - e_HA_gas) * HARTREE2KCAL
dG_solv_A  = (e_A_solv - e_A_gas) * HARTREE2KCAL

dG_aq = dG_gas + dG_solv_A + DG_SOLV_HPLUS - dG_solv_HA
RT_ln10 = 1.987e-3 * 298.15 * np.log(10)
pKa = dG_aq / RT_ln10

print("Acetic acid pKa (B3LYP/def2-SVP + ddCOSMO):")
print(f"  ΔG_gas (deprot)  = {dG_gas:+8.1f} kcal/mol")
print(f"  ΔG_solv(HA)      = {dG_solv_HA:+8.2f} kcal/mol")
print(f"  ΔG_solv(A⁻)      = {dG_solv_A:+8.2f} kcal/mol")
print(f"  ΔG_solv(H⁺) exp  = {DG_SOLV_HPLUS:+8.1f} kcal/mol")
print(f"  ΔG_aq (deprot)   = {dG_aq:+8.1f} kcal/mol")
print(f"  Computed pKa     = {pKa:.1f}  (experimental: 4.76)")
print("\nNote: Errors arise from geometry approximations and missing")
print("thermal corrections. Systematic pKa protocols reduce error to ~0.5 units.")

## 2. Explicit vs Implicit Solvation

| Model | Pros | Cons |
|-------|------|------|
| **Implicit (PCM/COSMO)** | Fast; no conformational sampling | No H-bonds; poor for multiply charged anions |
| **Explicit (QM/MM)** | Correct first-shell structure | Needs MD sampling; expensive |
| **Cluster + PCM** | First-shell explicit + bulk dielectric | Need cluster size convergence test |

**Rule of thumb**: Add 1–4 explicit water molecules for charged species before applying PCM.

## 🔬 Research Connection

Solvation models are central to computational chemistry and drug discovery:

- **Log P and solubility**: Aqueous and lipid solvation free energies predict partition
  coefficients (log P), a key pharmacokinetic descriptor.
- **pKa prediction**: Accurate pKa values guide ionisation state assignment in protein
  active sites and ADMET property prediction.
- **Electrochemistry**: Redox potentials in solution require ΔG_solv of oxidised and
  reduced species; critical for battery electrolyte and photoredox catalyst screening.
- **Atmospheric chemistry**: Henry's law constants and aqueous-phase reaction rates
  depend on ΔG_solv of trace gas molecules.

## 📋 Summary

| Concept | Key Equation / Fact |
|---------|---------------------|
| Born model | $\Delta G = -\frac{q^2}{2a}(1-1/\varepsilon)$ |
| PCM | Molecular cavity; apparent surface charges on $\partial\Omega$ |
| ddCOSMO | Domain-decomposition COSMO; $O(N)$ scaling |
| ΔG_solv | $E_{\rm solv} - E_{\rm gas}$ (negative = stabilized by solvent) |
| pKa cycle | $\Delta G_{\rm aq} = \Delta G_{\rm gas} + \Delta G_{\rm solv}(A^-) + \Delta G_{\rm solv}(H^+) - \Delta G_{\rm solv}(HA)$ |
| ε(water) | 78.4 at 25°C; large stabilisation of ions |

## 📝 Exercises

1. **Halide acids**: Compute ΔG_solv for HF, HCl in water. How does polarity affect solvation?

2. **Dielectric scan**: Repeat the ε-dependence for NH₃. At what ε does ΔG_solv saturate?
   Explain using the Born model.

3. **Explicit water**: Add one explicit water molecule H-bonded to NH₃ and recompute ΔG_solv
   using the cluster+PCM approach. Does the first solvation shell change the result significantly?

4. **Neutral vs ionic**: Compare ΔG_solv for acetic acid (charge=0) vs acetate (charge=-1).
   Which is more stabilised? Explain using the Born model's $q^2$ dependence.

5. **pKa of formic acid**: Apply the same thermodynamic cycle to formic acid (HCOOH,
   experimental pKa = 3.75). How does your result compare?